```text
Policy
├── Import & Setup
├── Define policies
└── Quantify impact
```

#### Import & Setup 

In [1]:
# Init work dir

from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
# Standard modules

import pandas as pd

# Framework

from src.feature_engineering import create_target_def12
from src.metrics import my_metrics
from src.modelling import apply_pipe, apply_pipe_LGD, estimate_capital
from src.preprocessing import my_input_load

# Configuration

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)
from sklearn import set_config; set_config(transform_output="pandas")

In [3]:
# load input file

portfolio = my_input_load(2021, 2021)
portfolio, dr_summary = create_target_def12(portfolio)

LoanDate range:     2021-01-01 00:00:00 2021-12-31 00:00:00
+12 months:         2022-01-01 00:00:00 2022-12-31 00:00:00
DefaultDate range:  2021-04-16 00:00:00 2023-10-13 00:00:00


LoanYear,num_loans,num_defaults,default_rate
2021,30266,3694,0.122051


In [4]:
# load PD, LGD pipelines

import joblib
pipeline_pd  = joblib.load("../models/PIPELINE_PD_ver_012.pkl")
pipeline_lgd = joblib.load("../models/PIPELINE_LGD_Amount.pkl")

# estimate PD, LGD

portfolio = apply_pipe(portfolio, pipeline_pd)
portfolio = apply_pipe_LGD(portfolio, pipeline_lgd)

## Define Policies

In [5]:
# Policy 1: PD based (PD < 0.20)

policy_1 = portfolio[portfolio["PD"]<0.20]

# Policy 2: Exposure based (Amount < 7.5k)

policy_2 = portfolio[portfolio["Amount"]<7500]

# Policy 3: Combined (PD < 0.20 and Amount < 7.5k)

policy_3 = portfolio[(portfolio["PD"]<0.20) & (portfolio["Amount"]<7500)]

# Policies

policies = {"POLICY 1": policy_1, "POLICY 2": policy_2, "POLICY 3": policy_3,}

## Quantify Impact

In [8]:
# impact: metrics + capital

policy_metrics = {}

for name, policy in policies.items():

    policy_metrics[name] = my_metrics(
        policy["default12"],
        policy["PD"],
        exposure=policy["Amount"],
        dataset_name=name,
    )

    _, loss_summary, el_summary, _ = estimate_capital(policy, method="monte-carlo", column_lgd="LGD", show_plot=False)
    el = el_summary["EL (MC-simulated)"]
    ec = loss_summary["Economic Capital"]
    print(f"Expected Loss    : {el:,.0f}")
    print(f"Economic Capital : {ec:,.0f}")
    

----------------------------------------
POLICY 1
----------------------------------------
AUC         : 0.6356
KS          : 0.2013
Observed DR : 9.21%
Mean PD     : 12.48%
Brier       : 0.0832
Exposure    : 59,068,472
----------------------------------------
Expected Loss    : 1,093,558
Economic Capital : 81,118
----------------------------------------
POLICY 2
----------------------------------------
AUC         : 0.6797
KS          : 0.2593
Observed DR : 11.92%
Mean PD     : 15.30%
Brier       : 0.1011
Exposure    : 66,052,220
----------------------------------------
Expected Loss    : 1,576,933
Economic Capital : 85,806
----------------------------------------
POLICY 3
----------------------------------------
AUC         : 0.6363
KS          : 0.2037
Observed DR : 9.17%
Mean PD     : 12.38%
Brier       : 0.0828
Exposure    : 47,892,899
----------------------------------------
Expected Loss    : 875,768
Economic Capital : 68,295
